# VCD — calibration de la suite de conformité sur un GPU gratuit

Colab : *Exécution → Modifier le type d'exécution → T4 GPU*. Kaggle : *Settings → Accelerator → GPU T4 x2 ou P100*.

La cellule suivante clone le dépôt public, installe deux dépendances légères et lance la suite en mode rapide (environ une minute). Le produit (T4.PCIE.1, P100.PCIE.1…) est déduit de la machine. L'objectif est de vérifier que l'identité détectée est juste et que les chiffres sont cohérents avec les performances connues du GPU.


In [ ]:
!rm -rf vcd && git clone -q https://github.com/benjamindarmian/vcd-protocol.git vcd
!cd vcd && pip -q install nvidia-ml-py cryptography && python -m oracle.run --detect --purpose calibration --mode quick --skip network --out receipt.json 2>&1 | tail -40 || true


## Lire et vérifier le reçu du mode rapide


In [ ]:
import json, subprocess
print(subprocess.run(['python','-m','oracle.verify','receipt.json'],cwd='vcd',capture_output=True,text=True).stdout)
r = json.load(open('vcd/receipt.json'))
print(json.dumps({'identite': {k: r['identity'][k] for k in ('gpu_count','family','form','driver_version','cuda_version')},
                  'gpus': [g.get('name') for g in r['identity']['gpus']],
                  'compute_tflops': r['measurements']['compute'].get('min_tflops'),
                  'hbm_gbs': r['measurements']['hbm'].get('min_gbs'),
                  'stability': r['measurements']['stability'].get('max_degradation_pct'),
                  'verdict': r['verdict']}, indent=1, ensure_ascii=False))


## Mode complet : 15 s de calcul, 5 minutes de charge pour la stabilité

Il suffit d'exécuter la cellule suivante (environ 6 minutes). Elle réutilise le dépôt déjà cloné.


In [ ]:
!cd vcd && python -m oracle.run --detect --purpose calibration --skip network --out receipt_full.json 2>&1 | tail -40 || true


In [ ]:
import json, subprocess
print(subprocess.run(['python','-m','oracle.verify','receipt_full.json'],cwd='vcd',capture_output=True,text=True).stdout)
r = json.load(open('vcd/receipt_full.json'))
st = r['measurements']['stability']
print(json.dumps({'compute_tflops': r['measurements']['compute'].get('min_tflops'),
                  'hbm_gbs': r['measurements']['hbm'].get('min_gbs'),
                  'storage_write_mbs': r['measurements']['storage'].get('write_mbs'),
                  'stabilite': {k: st.get(k) for k in ('seconds','max_degradation_pct','max_thermal_share','max_power_cap_share')},
                  'par_gpu': st.get('per_device'),
                  'verdict': r['verdict']}, indent=1, ensure_ascii=False))
